# Within-Chip SD Prediction Model

**CRISP-DM Phase 4 — Modeling**  
Predicting the average Scanner Density (`sd_mean`) per chip from waveform parameters.  
This answers: *what ink density level will this setting actually produce?*

See `documentation/why-sd-prediction.md` for why predicting SD directly is more fundamental than predicting std.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_absolute_error

try:
    from xgboost import XGBRegressor
    XGBOOST_AVAILABLE = True
except Exception:
    XGBOOST_AVAILABLE = False
    print('XGBoost not available — skipping.')

## 1. Load and aggregate
*CRISP-DM Phase 3 — Data Preparation*

Same aggregation as model 1, but target is `sd_mean` — the average ink density deposited per chip row.

In [ ]:
df = pd.read_parquet('../../data/processed/waveform_tuning_row_summary.parquet')

condition_cols = ['Color$', 'HeadIdx#', 'V', 'F_r', 'dt2', 'Coverage#']
agg = df.groupby(condition_cols)[['sd_mean', 'sd_std']].mean().reset_index()

print(f'Raw rows:          {len(df):,}')
print(f'After aggregation: {len(agg):,} conditions')
print(f'sd_mean range: {agg["sd_mean"].min():.5f} – {agg["sd_mean"].max():.5f}')

## 2. Feature engineering
*CRISP-DM Phase 3 — Data Preparation*

In [ ]:
color_dummies = pd.get_dummies(agg['Color$'], prefix='color', drop_first=False)
base = agg[['V', 'F_r', 'dt2', 'Coverage#']].copy()
base['V_x_Fr']         = agg['V'] * agg['F_r']
base['dt2_x_coverage'] = agg['dt2'] * agg['Coverage#']
base['V_sq']           = agg['V'] ** 2
base['Fr_sq']          = agg['F_r'] ** 2
X = pd.concat([base, color_dummies], axis=1)
y = agg['sd_mean']
print(f'Features: {X.columns.tolist()}')

## 3. Train / test split
*CRISP-DM Phase 4 — Modeling: Generate Test Design*

Split by HeadIdx#: train on chips 1–24, test on chips 25–30.

In [ ]:
train_mask = agg['HeadIdx#'] <= 24
X_train, X_test = X[train_mask], X[~train_mask]
y_train, y_test = y[train_mask], y[~train_mask]
print(f'Train: {len(X_train):,} rows')
print(f'Test:  {len(X_test):,} rows')

## 4. Train and evaluate models
*CRISP-DM Phase 4 — Modeling: Build Model*

In [ ]:
models = {
    'Linear Regression': LinearRegression(),
    'Random Forest':     RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1),
}
if XGBOOST_AVAILABLE:
    models['XGBoost'] = XGBRegressor(n_estimators=200, learning_rate=0.05, max_depth=5, random_state=42, verbosity=0)

results = {}
for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    results[name] = {'R2': r2_score(y_test, y_pred), 'MAE': mean_absolute_error(y_test, y_pred), 'model': model, 'y_pred': y_pred}

summary = pd.DataFrame({k: {'R²': v['R2'], 'MAE': v['MAE']} for k, v in results.items()}).T
print(summary.round(4))

## 5. Predicted vs actual
*CRISP-DM Phase 5 — Evaluation*

In [ ]:
n = len(results)
fig, axes = plt.subplots(1, n, figsize=(5*n, 4))
if n == 1: axes = [axes]
for ax, (name, res) in zip(axes, results.items()):
    ax.scatter(y_test, res['y_pred'], alpha=0.3, s=5)
    lims = [y_test.min(), y_test.max()]
    ax.plot(lims, lims, 'r--', linewidth=1)
    ax.set_xlabel('Actual sd_mean')
    ax.set_ylabel('Predicted sd_mean')
    ax.set_title(f'{name}\nR²={res["R2"]:.3f}  MAE={res["MAE"]:.5f}')
plt.tight_layout()
plt.show()

## 6. Use both models together
*CRISP-DM Phase 5 — Evaluation*

Combine this model with model 1 (sd_std) to get the full picture per condition:  
**sd_mean** = what ink density level you will get  
**sd_std** = how uniform the nozzles will be at that density

In [ ]:
best_name  = max(results, key=lambda k: results[k]['R2'])
best_model = results[best_name]['model']
print(f'Best model: {best_name}  (R²={results[best_name]["R2"]:.4f})')

# Manual prediction — change these values
V = 20.0; F_r = 1.26; dt2 = -1100; coverage = 10.0; color = 'C'

custom = pd.DataFrame([{'V': V, 'F_r': F_r, 'dt2': dt2, 'Coverage#': coverage, 'Color$': color}])
cd = pd.get_dummies(custom['Color$'], prefix='color', drop_first=False)
b = custom[['V','F_r','dt2','Coverage#']].copy()
b['V_x_Fr'] = custom['V']*custom['F_r']
b['dt2_x_coverage'] = custom['dt2']*custom['Coverage#']
b['V_sq'] = custom['V']**2
b['Fr_sq'] = custom['F_r']**2
Xc = pd.concat([b, cd], axis=1).reindex(columns=X_train.columns, fill_value=0)
pred_mean = best_model.predict(Xc)[0]

match = agg[(agg['V']==V)&(agg['F_r']==F_r)&(agg['dt2']==dt2)&(agg['Coverage#']==coverage)&(agg['Color$']==color)]
print(f'Settings: V={V}, F_r={F_r}, dt2={dt2}, Coverage={coverage}, Color={color}')
print(f'Predicted sd_mean: {pred_mean:.6f}')
if len(match):
    print(f'Actual    sd_mean: {match["sd_mean"].values[0]:.6f}  (error: {abs(pred_mean - match["sd_mean"].values[0]):.6f})')
else:
    print('Actual: not in dataset — prediction only')